In [24]:
import pandas as pd

from config import RAW_DIR, CLEAN_DB_PATH
from generate_data import main as generate_raw_data
from pipeline import (
    extract,
    clean_customers,
    clean_products,
    clean_orders,
    load_to_sqlite,
)

In [10]:
# Run this cell if you want to regenerate the random CSVs
generate_raw_data()
RAW_DIR

Generated 550 customers, 110 products, 1050 orders in C:\Users\Przemek\etl-pipeline\data\raw


WindowsPath('C:/Users/Przemek/etl-pipeline/data/raw')

In [25]:
raw_customers, raw_products, raw_orders = extract()

raw_customers.head()

,customer_id,first_name,last_name,email,phone,country,created_at
0,1,Daniel,Figueroa,jeffreydoyle@example.net,001-581-896-0013x3890,Cyprus,2025-07-13
1,2,Bridget,Pacheco,blakeerik@example.com,942-335-1161x55940,Uzbekistan,2025-02-15
2,3,Christian,Carter,barbara10@example.net,441.731.6475,Guinea,08-10-2025
3,4,Elizabeth,Miles,lynchgeorge@example.net,527.264.8350,Sri Lanka,2024-08-13
4,5,Richard,Jones,NaN,724.523.8849x696,Burundi,2024-11-04


In [26]:
raw_products.head()

,product_id,name,category,price,currency,in_stock
0,1,Effect,Books,392.98,USD,273
1,2,Exist,Home,123.39,USD,462
2,3,Bed,Miscellaneous,418.38,USD,347
3,4,Over,Home,473.83,USD,236
4,5,Other,Home,385.87,USD,257


In [13]:
raw_orders.head()

,order_id,customer_id,product_id,order_date,status,quantity,unit_price,total_amount
0,1,349,11,2025-06-24,delivered,5,226.14,1130.70
1,2,58,118,2025-04-13,shipped,5,105.07,525.35
2,3,267,88,02-05-2025,cancelled,2,359.87,719.74
3,4,229,59,2025-04-20,cancelled,2,259.06,"518,12"
4,5,530,40,2025-12-18,pending,1,200.99,200.99


In [27]:
customers = clean_customers(raw_customers)

customers.head()

c:\Users\Przemek\etl-pipeline\pipeline.py:32: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  "United States of America": "US",


,customer_id,first_name,last_name,email,phone,country,created_at
0,1,Daniel,Figueroa,jeffreydoyle@example.net,001-581-896-0013x3890,Cyprus,2025-07-13
1,2,Bridget,Pacheco,blakeerik@example.com,942-335-1161x55940,Uzbekistan,2025-02-15
2,3,Christian,Carter,barbara10@example.net,441.731.6475,Guinea,NaN
3,4,Elizabeth,Miles,lynchgeorge@example.net,527.264.8350,Sri Lanka,2024-08-13
6,7,Eric,Smith,teresa28@example.org,+1-948-293-2528x8095,Bouvet Island (Bouvetoya),2025-02-04


In [28]:
products = clean_products(raw_products)

products.head()

,product_id,name,category,price,currency,in_stock
0,1,Effect,Books,392.98,USD,273
1,2,Exist,Home,123.39,USD,462
2,3,Bed,Miscellaneous,418.38,USD,347
3,4,Over,Home,473.83,USD,236
4,5,Other,Home,385.87,USD,257


In [29]:
orders = clean_orders(raw_orders, customers=customers, products=products)

orders.head()

c:\Users\Przemek\etl-pipeline\pipeline.py:91: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  


,order_id,customer_id,product_id,order_date,status,quantity,unit_price,total_amount
0,1,349,11,2025-06-24,delivered,5,226.14,1130.70
3,4,229,59,2025-04-20,cancelled,2,259.06,518.12
5,6,24,90,2025-06-24,pending,1,272.88,272.88
7,8,196,49,2025-09-12,shipped,4,44.72,178.88
9,10,153,93,2025-03-06,cancelled,5,214.94,1074.70


In [30]:
load_to_sqlite(customers, products, orders)
CLEAN_DB_PATH

WindowsPath('C:/Users/Przemek/etl-pipeline/cleaned.db')

In [31]:
import sqlite3

conn = sqlite3.connect(CLEAN_DB_PATH)


In [39]:

pd.read_sql("SELECT * FROM customers", conn).head(50)

,customer_id,first_name,last_name,email,phone,country,created_at
0,1,Daniel,Figueroa,jeffreydoyle@example.net,+15818960013,Cyprus,2025-07-13
1,2,Bridget,Pacheco,blakeerik@example.com,NaN,Uzbekistan,2025-02-15
2,3,Christian,Carter,barbara10@example.net,NaN,Guinea,NaN
3,4,Elizabeth,Miles,lynchgeorge@example.net,NaN,Sri Lanka,2024-08-13
4,7,Eric,Smith,teresa28@example.org,+19482932528,Bouvet Island (Bouvetoya),2025-02-04
5,8,Michael,Galloway,john39@example.org,NaN,Croatia,2026-02-23
6,9,Amber,Blair,jenniferross@example.net,+15575871331,Albania,2026-02-23
7,10,Jean,Brown,whiteheadmichele@example.org,NaN,Palestinian Territory,2024-04-28
8,11,Michael,Jenkins,nuneztracey@example.org,NaN,Cocos (Keeling) Islands,2025-02-01
9,12,Steven,Baxter,ramosmichelle@example.net,NaN,Tonga,NaN
